# Ingesting dimension data into bronze layer
Reading raw CSVs from the volume and writing them as Delta tables in `quickbite.bronze`.
No cleaning at this stage, we are storing exactly what arrived plus metadata columns.

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType, DateType
import pyspark.sql.functions as F

catalog_name = 'quickbite'

## Restaurants

In [0]:
restaurant_schema = StructType([
    StructField("restaurant_id",  StringType(), False),
    StructField("name",           StringType(), True),
    StructField("city",           StringType(), True),
    StructField("cuisine_type",   StringType(), True),
    StructField("rating",         FloatType(),  True),
])

raw_path = "/Volumes/quickbite/source_data/raw/restaurants/"

df = (spark.read
      .option("header", "true")
      .option("delimiter", ",")
      .schema(restaurant_schema)
      .csv(raw_path)
      .withColumn("_source_file",  F.col("_metadata.file_path"))
      .withColumn("_ingested_at",  F.current_timestamp()))

display(df.limit(5))

In [0]:
df.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.bronze.brz_restaurants")

## Customers

In [0]:
customer_schema = StructType([
    StructField("customer_id",  StringType(), False),
    StructField("name",         StringType(), True),
    StructField("phone",        StringType(), True),
    StructField("city",         StringType(), True),
    StructField("signup_date",  StringType(), True),   # kept as string, cleaned in silver
])

raw_path = "/Volumes/quickbite/source_data/raw/customers/"

df = (spark.read
      .option("header", "true")
      .option("delimiter", ",")
      .schema(customer_schema)
      .csv(raw_path)
      .withColumn("_source_file",  F.col("_metadata.file_path"))
      .withColumn("_ingested_at",  F.current_timestamp()))

display(df.limit(5))

In [0]:
df.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.bronze.brz_customers")

## Delivery Agents

In [0]:
agent_schema = StructType([
    StructField("agent_id",      StringType(), False),
    StructField("name",          StringType(), True),
    StructField("vehicle_type",  StringType(), True),
    StructField("city",          StringType(), True),
])

raw_path = "/Volumes/quickbite/source_data/raw/delivery_agents/"

df = (spark.read
      .option("header", "true")
      .option("delimiter", ",")
      .schema(agent_schema)
      .csv(raw_path)
      .withColumn("_source_file",  F.col("_metadata.file_path"))
      .withColumn("_ingested_at",  F.current_timestamp()))

display(df.limit(5))

In [0]:
df.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.bronze.brz_delivery_agents")

## Menu Items

In [0]:
menu_schema = StructType([
    StructField("item_id",     StringType(), False),
    StructField("item_name",   StringType(), True),
    StructField("category",    StringType(), True),
    StructField("base_price",  StringType(), True),   # string may have currency symbols
])

raw_path = "/Volumes/quickbite/source_data/raw/menu_items/"

df = (spark.read
      .option("header", "true")
      .option("delimiter", ",")
      .schema(menu_schema)
      .csv(raw_path)
      .withColumn("_source_file",  F.col("_metadata.file_path"))
      .withColumn("_ingested_at",  F.current_timestamp()))

display(df.limit(5))

In [0]:
df.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.bronze.brz_menu_items")

## Date Dimension

In [0]:
date_schema = StructType([
    StructField("date",         StringType(),  True),
    StructField("year",         IntegerType(), True),
    StructField("month",        IntegerType(), True),
    StructField("month_name",   StringType(),  True),
    StructField("day_name",     StringType(),  True),
    StructField("quarter",      IntegerType(), True),
    StructField("week_of_year", IntegerType(), True),   # may be negative, fixed in silver
    StructField("is_weekend",   IntegerType(), True),
])

raw_path = "/Volumes/quickbite/source_data/raw/date/"

df = (spark.read
      .option("header", "true")
      .option("delimiter", ",")
      .schema(date_schema)
      .csv(raw_path)
      .withColumn("_source_file",  F.col("_metadata.file_path"))
      .withColumn("_ingested_at",  F.current_timestamp()))

display(df.limit(5))

In [0]:
df.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.bronze.brz_date")

## Sanity check of row counts

In [0]:
tables = ["brz_restaurants","brz_customers","brz_delivery_agents","brz_menu_items","brz_date"]
for t in tables:
    cnt = spark.sql(f"SELECT count(*) as cnt FROM quickbite.bronze.{t}").collect()[0]["cnt"]
    print(f"{t:30s}  {cnt:>6} rows")